# Ejercicio 5 — Incorporación de datos de 2024

Se amplía el sistema para trabajar con 2024 además de 2026, obteniendo los datos directamente de la fuente original (CloudFront de la TLC). Consultas de validación: [`sql/05_incorporacion_2024.sql`](../sql/05_incorporacion_2024.sql).

## 5.1 Modificación del sistema de descarga

El script original tenía el año fijo en una constante (`ANIO = 2026`) que se usaba para construir el nombre del archivo, la URL, la ruta local y el ciclo de meses. Se reemplazó por un parámetro (detalle en [`docs/ejercicio2.md`](../docs/ejercicio2.md)):

* `ANIOS = (2024, 2025, 2026)` es la lista de años por defecto;
* `--anio` acepta uno o varios años (`--anio 2024 2026`);
* `construir_nombre`, `construir_url`, `ruta_destino` y `descargar` reciben el año como argumento.

Para incorporar 2024 se ejecutó:

```bash
docker compose exec lab python scripts/download_data.py --anio 2024 2026
```

## 5.2–5.4 Ejecución de la descarga

Salida resumida de esa ejecución (la salida completa lista los 48 meses):

```text
=== YELLOW 2024 ===
  2024-01  listo (47.6 MiB) -> data/raw/yellow/2024/yellow_tripdata_2024-01.parquet
  ...                                  (12 meses, 47.6 a 61.4 MiB cada uno)
=== GREEN 2024 ===
  2024-01  listo (1.3 MiB) -> data/raw/green/2024/green_tripdata_2024-01.parquet
  ...                                  (12 meses, 1.2 a 1.4 MiB cada uno)
=== YELLOW 2026 ===
  2026-01  ya existe, se omite
  ...                                  (2026-01 a 2026-08)
  2026-09  aun no publicado por la TLC
  ...                                  (2026-09 a 2026-12)
=== GREEN 2026 ===                     (igual que amarillos)
RESUMEN
  descargados   : 24
  ya existian   : 16
  no publicados : 8
  fallidos      : 0
```

* **5.2** Los 16 archivos de 2026 se conservaron: el script nunca borra archivos y solo escribe en la ruta del mes que descarga.
* **5.3** No se volvieron a descargar: antes de consultar el servidor, el script comprueba si `data/raw/<tipo>/<anio>/<archivo>` existe y no está vacío (`ya existe, se omite`).
* **5.4** Se descargaron los 24 archivos de 2024 (12 meses × 2 tipos), sin fallos.

In [1]:
import sys

import pandas as pd

sys.path.append("../scripts")
from lab import conectar_parquet, consultas, mostrar

pd.set_option("display.max_columns", None)
con = conectar_parquet()
Q = consultas("05_incorporacion_2024.sql")

## 5.5 Verificación de los archivos nuevos

In [2]:
mostrar(con, Q["5.5_archivos_por_anio"])

-- Objetivo: verificar que archivos hay por tipo de taxi y anio (2026 debe conservarse, 2024 debe estar completo).
-- Fuente: nombres de archivo de data/raw/*/*/*.parquet (glob).
SELECT regexp_extract(file, 'raw/(\w+)/', 1)                         AS tipo_taxi,
       CAST(regexp_extract(file, 'raw/\w+/(\d{4})/', 1) AS INTEGER)  AS anio,
       count(*)                                                      AS archivos,
       min(regexp_extract(file, '(\d{4}-\d{2})\.parquet$', 1))       AS primer_mes,
       max(regexp_extract(file, '(\d{4}-\d{2})\.parquet$', 1))       AS ultimo_mes
FROM glob('data/raw/*/*/*.parquet')
WHERE anio IN (2024, 2026)
GROUP BY ALL
ORDER BY tipo_taxi, anio;


,tipo_taxi,anio,archivos,primer_mes,ultimo_mes
0,green,2024,12,2024-01,2024-12
1,green,2026,8,2026-01,2026-08
2,yellow,2024,12,2024-01,2024-12
3,yellow,2026,8,2026-01,2026-08


In [3]:
mostrar(con, Q["5.5_archivos_legibles"])

-- Objetivo: comprobar que cada archivo de 2024 es un Parquet legible y con registros.
-- Un archivo truncado no tiene pie (footer) valido y esta consulta fallaria.
-- Fuente: metadatos de data/raw/*/2024/*.parquet (parquet_file_metadata).
SELECT regexp_extract(file_name, 'raw/(\w+)/', 1)  AS tipo_taxi,
       count(*)                                    AS archivos_legibles,
       min(num_rows)                               AS registros_min_por_archivo,
       sum(num_rows)::BIGINT                       AS registros
FROM parquet_file_metadata('data/raw/*/2024/*.parquet')
GROUP BY tipo_taxi
ORDER BY tipo_taxi;


,tipo_taxi,archivos_legibles,registros_min_por_archivo,registros
0,green,12,51771,660218
1,yellow,12,2964624,41169720


**Resultado:** 2024 tiene los 12 meses de cada tipo y 2026 conserva sus 8 meses. Los 24 archivos de 2024 son Parquet legibles: un archivo truncado no tendría un pie válido y `parquet_file_metadata` fallaría. Suman 41,169,720 viajes amarillos y 660,218 verdes, y ningún archivo está vacío (el mes con menos registros tiene 2.96 millones de amarillos y 51,771 verdes).

## 5.6 Consulta conjunta de 2024 y 2026

In [4]:
mostrar(con, Q["5.6_resumen_por_anio"])

-- Objetivo: consultar 2024 y 2026 juntos con la misma vista, sin cambios en ella.
-- Fuente: vista `viajes_validos` sobre data/raw/*/{2024,2026}/*.parquet.
SELECT anio,
       tipo_taxi,
       count(DISTINCT mes)                                   AS meses,
       count(*)                                              AS viajes,
       round(avg(trip_distance), 2)                          AS distancia_prom_mi,
       round(avg(total_amount), 2)                           AS total_prom,
       round(100.0 * count_if(forma_pago = 'Tarjeta') / count(*), 1) AS pct_tarjeta
FROM viajes_validos
WHERE anio IN (2024, 2026)
GROUP BY anio, tipo_taxi
ORDER BY tipo_taxi, anio;


,anio,tipo_taxi,meses,viajes,distancia_prom_mi,total_prom,pct_tarjeta
0,2024,green,12,618838,2.93,24.06,68.8
1,2026,green,8,316783,3.29,25.29,66.6
2,2024,yellow,12,39666389,3.42,28.63,76.0
3,2026,yellow,8,28212241,3.51,30.24,65.4


In [5]:
mostrar(con, Q["5.6_mismo_mes_2024_vs_2026"])

-- Objetivo: comparar los meses que existen en ambos anios (enero-agosto).
-- Fuente: vista `viajes_validos`, anios 2024 y 2026.
SELECT tipo_taxi,
       mes,
       count_if(anio = 2024)::BIGINT                         AS viajes_2024,
       count_if(anio = 2026)::BIGINT                         AS viajes_2026,
       round(100.0 * (viajes_2026 - viajes_2024) / viajes_2024, 1) AS variacion_pct
FROM viajes_validos
WHERE anio IN (2024, 2026)
  AND mes IN (SELECT DISTINCT mes FROM viajes WHERE anio = 2026)
GROUP BY tipo_taxi, mes
ORDER BY tipo_taxi, mes;


,tipo_taxi,mes,viajes_2024,viajes_2026,variacion_pct
0,green,1,53110,38062,-28.3
1,green,2,50246,35092,-30.2
2,green,3,53917,41648,-22.8
3,green,4,52769,41545,-21.3
4,green,5,57271,42309,-26.1
5,green,6,51489,41519,-19.4
6,green,7,48305,38565,-20.2
7,green,8,48412,38043,-21.4
8,yellow,1,2866581,3514484,22.6
9,yellow,2,2898578,3208135,10.7


In [6]:
mostrar(con, Q["5.6_columnas_por_anio"])

-- Objetivo: ver como union_by_name resuelve las diferencias de esquema entre anios.
-- Fuente: vista `viajes`, anios 2024 y 2026.
SELECT anio,
       tipo_taxi,
       count(*)                                                     AS registros,
       round(100.0 * count(cbd_congestion_fee) / count(*), 1)       AS pct_con_cbd_congestion_fee,
       round(100.0 * count(request_source) / count(*), 1)           AS pct_con_request_source,
       round(100.0 * count(passenger_count) / count(*), 1)          AS pct_con_passenger_count
FROM viajes
WHERE anio IN (2024, 2026)
GROUP BY anio, tipo_taxi
ORDER BY tipo_taxi, anio;


,anio,tipo_taxi,registros,pct_con_cbd_congestion_fee,pct_con_request_source,pct_con_passenger_count
0,2024,green,660218,0.0,0.0,96.3
1,2026,green,337114,100.0,5.7,85.5
2,2024,yellow,41169720,0.0,0.0,90.1
3,2026,yellow,29703355,100.0,9.8,74.0


**Resultado:** la misma vista `viajes_validos` devuelve ambos años en una sola consulta, sin cambios en su definición. Comparando los mismos meses (enero–agosto), los viajes amarillos crecieron entre 6% y 23% de 2024 a 2026, mientras que los verdes cayeron entre 19% y 30%. El total promedio subió (amarillos de USD 28.63 a 30.24) y el pago con tarjeta bajó de 76% a 65%.

Las diferencias de esquema se resuelven solas con `union_by_name`: `cbd_congestion_fee` y `request_source` no existen en 2024 y quedan en NULL (0% de registros con dato), y `passenger_count` falta en 9.9% de los amarillos de 2024 frente a 26% en 2026.

## 5.7 ¿Hay que modificar las consultas anteriores?

**No.** Las consultas de los ejercicios anteriores siguen funcionando sin cambios:

* La vista `viajes` lee `data/raw/<tipo>/*/*.parquet`, así que la carpeta `2024/` entra sola en el glob, y `anio`/`mes` se derivan del nombre de cada archivo.
* Las consultas del Ejercicio 3 apuntan explícitamente a `data/raw/*/2026/`, y las del Ejercicio 4 filtran `anio = 2026`, porque ese era su alcance. Por eso devuelven exactamente lo mismo que antes. Para analizar 2024 basta con cambiar el valor del filtro o agrupar por `anio`; no hay que tocar fuentes ni rutas.

Comprobación con la pregunta P4 del Ejercicio 4, tal como está en `sql/04_analisis_exploratorio.sql` y luego solo cambiando el año:

In [7]:
Q4 = consultas("04_analisis_exploratorio.sql")["4.P4_amarillo_vs_verde"]
con.sql(Q4).df()   # sin cambios: mismo resultado que en el Ejercicio 4

,tipo_taxi,viajes,pct_viajes,distancia_prom_mi,tarifa_prom,total_prom,tarifa_por_milla,pct_tarifa_aeropuerto,pct_cobro_cbd,pct_despachados
0,green,316783,1.11,3.29,16.95,25.29,5.15,0.25,8.46,3.01
1,yellow,28212241,98.89,3.51,21.30,30.24,6.07,2.63,72.42,NaN


In [8]:
con.sql(Q4.replace("anio = 2026", "anio = 2024")).df()   # la misma consulta sobre 2024

,tipo_taxi,viajes,pct_viajes,distancia_prom_mi,tarifa_prom,total_prom,tarifa_por_milla,pct_tarifa_aeropuerto,pct_cobro_cbd,pct_despachados
0,green,618838,1.54,2.93,18.0,24.06,6.13,0.28,NaN,2.96
1,yellow,39666389,98.46,3.42,19.8,28.63,5.79,3.64,NaN,NaN


La única precaución es al interpretar columnas que no existían en un año: el porcentaje de viajes con cargo CBD en 2024 es NULL porque el cargo empezó en enero de 2025, no porque falten datos.

## 5.8 Consultas usadas para validar la incorporación

| Consulta (`sql/05_incorporacion_2024.sql`) | Objetivo | Fuente | Resultado |
|---|---|---|---|
| `5.5_archivos_por_anio` | Archivos por tipo y año | nombres de `data/raw/*/*/*.parquet` | 12 meses de 2024 y 8 de 2026 por tipo |
| `5.5_archivos_legibles` | Que cada archivo de 2024 sea un Parquet válido y no vacío | metadatos de `data/raw/*/2024/*.parquet` | 24 archivos legibles, 41.8 M registros |
| `5.6_resumen_por_anio` | Consultar ambos años juntos | vista `viajes_validos` | viajes y promedios por año y tipo |
| `5.6_mismo_mes_2024_vs_2026` | Comparar meses equivalentes | vista `viajes_validos` | amarillos +6% a +23%, verdes −19% a −30% |
| `5.6_columnas_por_anio` | Ver cómo se alinean esquemas distintos | vista `viajes` | columnas nuevas en NULL para 2024 |
| `4.P4_amarillo_vs_verde` (Ej. 4) | Comprobar que una consulta anterior no requiere cambios | vista `viajes_validos` | mismo resultado para 2026; funciona para 2024 |

## 5.9 Características del diseño que permiten incorporar archivos sin rehacer el análisis

1. **Ruta por convención** `data/raw/<tipo>/<anio>/<archivo original>`: el tipo y el año se deducen de la ruta, sin catálogos que mantener.
2. **Descarga idempotente y parametrizada:** el año es un argumento; los archivos existentes se omiten; los meses aún no publicados se reportan y se descargan en una ejecución posterior.
3. **Vistas sobre globs** (`*/*.parquet`): un archivo nuevo forma parte del análisis en cuanto existe en disco, sin paso de carga.
4. **`union_by_name`:** las columnas nuevas o ausentes de un año (por ejemplo, `cbd_congestion_fee` o `request_source`) no rompen la lectura.
5. **Lógica en capas:** `viajes` (lectura y unificación) → `viajes_validos` (reglas de calidad) → consultas de análisis. Una regla nueva se cambia en un solo lugar.
6. **Consultas sin años fijos donde no hace falta:** las del tablero (Ejercicio 7) agrupan por `anio`/`mes`, así que un año nuevo aparece como una serie más.